### **Methodology: Adapting Recipe Network Science to Unstructured Reviews**

**1. Conceptual Framework and Inspiration**
Our methodological approach was heavily inspired by the work of Kim and Chung in *"Tell Me What You Eat, and I Will Tell You Where You Come From"* [1]. In their research, the authors demonstrated that culinary culture could be mathematically modeled by constructing **Ingredient Networks** based on co-occurrence matrices. They successfully argued that while common ingredients (like salt or water) create noise, the structural identity of a cuisine is defined by unique, "authentic" connections (e.g., *soy sauce* co-occurring with *sesame oil*).

We adopted this framework but faced a significantly harder challenge: while Kim and Chung analyzed structured recipe databases with clear ingredient lists, we were tasked with mining **unstructured, noisy customer reviews**.

**2. From "Ingredients" to "Dishes": The NER Pipeline**
Since reviews rarely list raw ingredients, we adapted the authors' concept of "Ingredient Analysis" into **"Dish Analysis."** To extract structured entities from unstructured text, manual tagging (as used in early recipe research) was unfeasible.
Instead, we benchmarked fine-tuned BERT models on NVIDIA A40 GPU via RunPod. After testing `Dizex/FoodBaseBERT` (got low recall) and `deberta-v3-food` (got low precision), we selected **`Dizex/InstaFoodRoBERTa-NER`**. This model, fine-tuned on social media captions, effectively acted as our "automated parser", converting free text into the structured lists required for network analysis.

**3. Network Construction and "Backbone Extraction"**
Kim and Chung emphasized the importance of **"Backbone Extraction"** (Section III.B) to filter statistically insignificant links. We observed the same necessity: our initial raw network was a "hairball" dominated by global hubs.
Following the authors' logic regarding "Authenticity" (Table 4)—where they mathematically penalized ubiquitous ingredients to find region-specific ones—we implemented a **Discriminative Filtering Strategy**:
*   **Hub Removal:** We removed global "stop-food-words" (e.g., *chicken, rice, sauce*) which correspond to the "low authenticity" ingredients in the reference paper.
*   **Edge Pruning:** We applied a dynamic threshold to the co-occurrence weights, effectively replicating the backbone extraction technique to retain only the strongest semantic pairings (e.g., *Chips* $\leftrightarrow$ *Salsa*).

**4. Unsupervised Clustering and Validation**
While the reference paper used Hierarchical Clustering to group *countries*, we applied **Greedy Modularity Maximization** to group *dishes*. This unsupervised algorithm successfully partitioned the network into distinct communities without prior labeling.
The results validated the hypothesis proposed by Kim and Chung: food entities naturally segregate into clusters based on cultural usage.
*   Our **"Blue Cluster"** (containing *taco, burrito, queso*) shows Latin American/Mexican dishes patterns.
*   Our **"Yellow Cluster"** (containing *meatball, sphagetti, lasagna*) mirrors their findings on the distinctiveness of Itallian culinary profiles.

By adapting the network science approach from structured recipes to unstructured reviews, we successfully demonstrated that *e-Word-of-Mouth* data retains the same distinct cultural topology as professional recipe databases.


In [1]:
import pandas as pd
import numpy as np
import networkx as nx
from pyvis.network import Network
from sklearn.feature_extraction.text import CountVectorizer
from networkx.algorithms import community
import matplotlib.colors as mcolors

In [2]:
subset_10k = pd.read_pickle("data_atlanta/atlanta_full_roberta_raw.pkl")

In [3]:
import string
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords

# Initialize lemmatizer
lemmatizer = WordNetLemmatizer()
# We don't want to remove 'meat' or 'fish', but we might want to remove generic words like 'the'
stop_words = set(stopwords.words("english"))


def clean_food_entity(entity):
    """
    Normalizes a single food entity string.

    1. Lowercases the text.
    2. Removes punctuation (e.g., 'pizza!' -> 'pizza').
    3. Lemmatizes each word (e.g., 'burgers' -> 'burger').
    4. Removes stopwords to avoid 'the pasta' vs 'pasta'.

    Args:
        entity (str): Raw entity string (e.g., "Chicken Wings!")

    Returns:
        str: Normalized string (e.g., "chicken wing") or None if invalid.
    """
    if not isinstance(entity, str):
        return None

    # 1. Lowercase and strip
    entity = entity.lower().strip()

    # 2. Remove punctuation
    entity = entity.translate(str.maketrans("", "", string.punctuation))

    # 3. Tokenize to process individual words in the phrase
    words = entity.split()

    # 4. Lemmatize and remove stopwords
    cleaned_words = [
        lemmatizer.lemmatize(w)
        for w in words
        if w not in stop_words and len(w) > 1  # Filter single chars
    ]

    if not cleaned_words:
        return None

    # Rejoin (e.g., ['fried', 'rice'] -> "fried rice")
    return " ".join(cleaned_words)


def normalize_food_column(food_list):
    """
    Applies cleaning to a list of food entities and removes duplicates.
    """
    if not isinstance(food_list, list):
        return []

    normalized_list = []
    for item in food_list:
        clean_item = clean_food_entity(item)
        if clean_item:
            normalized_list.append(clean_item)

    # Return unique items only (set) to handle ["pizza", "Pizza!"] becoming ["pizza", "pizza"]
    return list(set(normalized_list))


# --- APPLY TO DATAFRAME ---

# Drop rows that are null in the original extraction or have empty text
subset_10k = subset_10k.dropna(subset=["text_for_clf"])

# Apply normalization
subset_10k["normalized_food"] = subset_10k["roberta_food"].apply(normalize_food_column)

# Filter out rows that have no foods left after normalization
df_analysis = subset_10k[subset_10k["normalized_food"].map(len) > 0].copy()

print(
    f"Original unique raw entities: {len(set([x for sub in subset_10k['roberta_food'] for x in sub]))}"
)
print(
    f"Normalized unique entities: {len(set([x for sub in df_analysis['normalized_food'] for x in sub]))}"
)
print("Example check:")
print(df_analysis[["roberta_food", "normalized_food"]].sample(5))

Original unique raw entities: 16647
Normalized unique entities: 10752
Example check:
                                            roberta_food  \
11456                         [mango kulfi, apollo fish]   
42226  [veggies, baklava, chicken, date breads., fala...   
39160                                   [hibachi, sushi]   
42610                                [chicken, biscuits]   
49255                                [chicken, chicken.]   

                                         normalized_food  
11456                         [mango kulfi, apollo fish]  
42226  [veggie, baklava, shawarma wrap, falafel wrap,...  
39160                                   [sushi, hibachi]  
42610                                 [biscuit, chicken]  
49255                                          [chicken]  


In [4]:
# ==========================================
# 1. CONFIGURATION
# ==========================================

# Extended Pastel Palette (High Contrast borders will be added)
PASTEL_PALETTE = [
    "#FFB3BA",  # Cherry Blossom
    "#BAFFC9",  # Mint
    "#BAE1FF",  # Sky Blue
    "#FFFFBA",  # Pastel Yellow
    "#FFDFBA",  # Peach
    "#E0BBE4",  # Lavender
    "#957DAD",  # Muted Purple
    "#FFC3A0",  # Deep Peach
    "#D5AAFF",  # Light Purple
    "#B5B9FF",  # Periwinkle
    "#85E3FF",  # Cyan
    "#FF9CEE",  # Pink
]

# The "Kill List" (Ingredient Stopwords) - SAME AS BEFORE
INGREDIENT_STOPWORDS = {
    "food",
    "place",
    "restaurant",
    "order",
    "menu",
    "dish",
    "meal",
    "dinner",
    "lunch",
    "chicken",
    "rice",
    "sauce",
    "cheese",
    "salad",
    "bread",
    "meat",
    "fry",
    "fries",
    "vegetable",
    "veggie",
    "drink",
    "water",
    "salt",
    "pepper",
    "sugar",
    "butter",
    "side",
    "appetizer",
    "entree",
    "dessert",
    "soup",
    "sandwich",
    "burger",
    "steak",
    "fish",
    "seafood",
    "beef",
    "pork",
    "shrimp",
    "pasta",
    "pizza",
    "wing",
    "taco",
    "sushi",
    "curry",
    "noodle",
    "egg",
    "bacon",
    "sausage",
    "potato",
    "corn",
    "bean",
    "onion",
    "tomato",
    "lettuce",
    "avocado",
    "chip",
    "salsa",
    "dip",
    "tea",
    "coffee",
    "beer",
    "wine",
    "cocktail",
    "margarita",
    "bar",
    "table",
    "server",
    "service",
    "time",
    "review",
    "star",
    "experience",
    "friend",
    "family",
    "people",
    "night",
    "day",
    "plate",
    "bowl",
    "cup",
    "glass",
    "bottle",
    "nugget",
    "lemonade",
    "hummus",
    "oil",
    "garlic",
    "ginger",
    "soy",
    "cream",
    "milk",
    "bun",
    "crust",
    "slice",
    "piece",
    "bit",
}

# ==========================================
# 2. DATA PREPARATION (Back to 160 Nodes)
# ==========================================

# Prepare counts
all_norm_foods = [
    food for sublist in df_analysis["normalized_food"] for food in sublist
]
food_counts = pd.Series(all_norm_foods).value_counts()

# Keep top 160 (More density)
clean_counts = food_counts[~food_counts.index.isin(INGREDIENT_STOPWORDS)]
top_entities = clean_counts.head(160).index.tolist()

print(f"Selected {len(top_entities)} specific dishes.")

# Filter Dataset
df_temp = df_analysis.copy()
df_temp["filtered_food"] = df_temp["normalized_food"].apply(
    lambda x: [item for item in x if item in top_entities]
)
df_temp = df_temp[df_temp["filtered_food"].map(len) > 1]


# Build Matrix
def dummy(doc):
    return doc


vectorizer = CountVectorizer(
    tokenizer=dummy,
    preprocessor=dummy,
    token_pattern=None,
    vocabulary=top_entities,
    binary=True,
)
X = vectorizer.fit_transform(df_temp["filtered_food"])
terms = vectorizer.get_feature_names_out()
co_occurrence_df = pd.DataFrame((X.T * X).toarray(), index=terms, columns=terms)
np.fill_diagonal(co_occurrence_df.values, 0)

# ==========================================
# 3. GRAPH & CLUSTERING
# ==========================================

G = nx.from_pandas_adjacency(co_occurrence_df)

# Pruning: Use a slightly lower threshold than last time to keep structure
# We keep edges that appear at least 2 times, but favor stronger ones
edges_to_keep = [(u, v) for u, v, d in G.edges(data=True) if d["weight"] >= 2]
G_clean = G.edge_subgraph(edges_to_keep).copy()
G_clean.remove_nodes_from(list(nx.isolates(G_clean)))

# Community Detection
# Resolution 1.4 was the "Magic Number" from the dark version
communities = community.greedy_modularity_communities(G_clean, resolution=1.4)

# Map Communities
community_map = {}
sorted_communities = sorted(communities, key=len, reverse=True)
# Only keep clusters with > 3 items to reduce noise
valid_nodes = []
for i, comm in enumerate(sorted_communities):
    if len(comm) > 3:
        valid_nodes.extend(comm)
        for node in comm:
            community_map[node] = i

G_final = G_clean.subgraph(valid_nodes).copy()
num_comms = len(set(community_map.values()))
print(f"Final Graph: {len(G_final.nodes())} nodes in {num_comms} clusters.")

# ==========================================
# 4. VISUALIZATION (White Background + Borders)
# ==========================================

net = Network(
    height="850px", width="100%", bgcolor="#ffffff", font_color="#333333", notebook=True
)

# Context Map (Ground Truth)
node_cuisine_map = {}
for food in G_final.nodes():
    mask = df_analysis["normalized_food"].apply(lambda x: food in x)
    if mask.sum() > 0:
        top_cat = df_analysis.loc[mask, "categoryName"].mode()
        node_cuisine_map[food] = top_cat[0] if not top_cat.empty else "Unknown"

for node in G_final.nodes():
    comm_id = community_map[node]
    color = PASTEL_PALETTE[comm_id % len(PASTEL_PALETTE)]

    degree = G_final.degree[node]
    size = 15 + (degree * 1.5)

    title_html = f"<b>{node.upper()}</b><br>Cuisine: {node_cuisine_map.get(node)}"

    # Adding BORDER is crucial for pastel on white
    net.add_node(
        node,
        label=node,
        title=title_html,
        color=color,
        size=size,
        font={"size": 18, "face": "Arial", "color": "#222222", "strokeWidth": 0},
        borderWidth=2,
        borderColor="#555555",
    )

for u, v, data in G_final.edges(data=True):
    weight = data["weight"]

    # Coloring logic: Colored if internal, Grey if external
    if community_map[u] == community_map[v]:
        edge_color = PASTEL_PALETTE[community_map[u] % len(PASTEL_PALETTE)]
        opacity = 0.6
        width = (weight / 2) + 1
        if width > 8:
            width = 8
    else:
        edge_color = "#cccccc"
        opacity = 0.2
        width = 1

    net.add_edge(u, v, value=width, color=edge_color, alpha=opacity)

# PHYSICS: Force Atlas 2 Based (The one that "Explodes" clusters nicely)
net.force_atlas_2based(
    gravity=-60,  # Repulsion
    central_gravity=0.005,  # Keep them vaguely together
    spring_length=100,
    spring_strength=0.08,
    damping=0.4,
    overlap=0,
)

# STABILIZATION: This pre-calculates the layout so it doesn't spin
net.toggle_stabilization(True)

net.show("atlanta_food_final_white.html")

Selected 160 specific dishes.
Final Graph: 158 nodes in 9 clusters.
atlanta_food_final_white.html
